# 🚀 Giorno 3 - Lab 5: Esercitazione — Costruisci il Tuo MVP con LangChain

Benvenuti all'ultimo laboratorio del **Giorno 3**!

Nei Lab 1-4 hai imparato:
- I blocchi fondamentali LangChain: `ChatOllama`, `@tool`, `ChatPromptTemplate`, `AgentExecutor` (Lab 1)
- Come costruire un **Agent + Tools** con `create_tool_calling_agent` (Lab 2)
- Come implementare **Human Approval** con `BaseCallbackHandler` (Lab 3)
- Come applicare **Agent-as-Tool** con `Tool.from_function()` (Lab 4)

Ora costruisci il tuo **MVP personale** partendo dal Project Canvas del Giorno 1.

---
### 🎯 Obiettivo
> Il tuo flow risponde correttamente ad almeno **3 richieste con esiti diversi**.

---
### 📋 Struttura in 6 Fasi
```
FASE 1 — Dati (15 min)           → file JSON/Markdown nel dominio del tuo caso d'uso
FASE 2 — Tool (20 min)           → funzioni @tool con docstring chiara
FASE 3 — System Prompt (10 min)  → ChatPromptTemplate con ruolo + esiti + sequenza
FASE 4 — AgentExecutor (10 min)  → assemblaggio con create_tool_calling_agent
FASE 5 — Test (15 min)           → 3 scenari con esiti diversi
FASE 6 — Estensioni (opz.)       → Human Approval o Agent-as-Tool
```

## Setup Comune (Esegui Sempre)

In [ ]:
import json
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.tools import tool, Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

DATA_DIR = Path("data")
llm      = ChatOllama(model="llama3.2", temperature=0)
print("✅ LangChain + ChatOllama pronti.")

---
## FASE 1 — Dati del Tuo Dominio

Definisci i dati su cui il tuo agente dovrà ragionare.
Puoi usare dizionari Python inline (prototipo rapido) oppure file nella cartella `data/`.

In [ ]:
# ============================================================
# FASE 1: DATI DEL TUO DOMINIO
# ============================================================

# Modifica con i dati del tuo caso d'uso
MIO_DATASET_1 = {
    "U001": {
        "nome": "Nome Utente 1",
        "ruolo": "...",
        "attributo_chiave": "...",   # es. budget, giorni ferie, livello priorità
    },
    "U002": {
        "nome": "Nome Utente 2",
        "ruolo": "...",
        "attributo_chiave": "...",
    }
}

MIA_POLICY_TEXT = """
# REGOLE DEL TUO DOMINIO

## Regola 1
...

## Regola 2
...
"""

print("📁 Dati definiti:")
print(f"   Dataset: {list(MIO_DATASET_1.keys())}")
print(f"   Policy:  {len(MIA_POLICY_TEXT)} caratteri")

---
## FASE 2 — Tool con `@tool`

**Ricorda:** la docstring deve rispondere a "quando deve essere chiamato questo tool?".
È quella che l'LLM legge per decidere quando invocarlo.

**MVP: almeno 2 tool.**

In [ ]:
# ============================================================
# FASE 2: DEFINISCI I TUOI TOOL CON @tool
# ============================================================

@tool
def mio_tool_1(parametro_1: str) -> dict:
    """TODO: scrivi QUANDO questo tool deve essere chiamato.
    Deve rispondere a: in quale situazione l'agente ha bisogno di questi dati?"""
    # TODO: implementa la logica
    return {"status": "OK", "risultato": f"Dati per {parametro_1}"}


@tool
def mio_tool_2(parametro_a: str) -> dict:
    """TODO: scrivi QUANDO questo tool deve essere chiamato."""
    # TODO: implementa la logica
    return {"status": "OK", "risultato": f"Ricerca per {parametro_a}"}


# Verifica schema generato automaticamente
for t in [mio_tool_1, mio_tool_2]:
    print(f"✅ {t.name}: {t.description[:60]}...")
    print(f"   Args: {t.args}")

---
## FASE 3 — System Prompt

**Checklist:**
- [ ] Ruolo specifico dell'agente
- [ ] Possibili esiti elencati esplicitamente
- [ ] Sequenza operativa (quali tool chiamare e in quale ordine)
- [ ] `Rispondi in italiano` come ultima istruzione

In [ ]:
# ============================================================
# FASE 3: SYSTEM PROMPT
# ============================================================

MIO_SYSTEM_PROMPT = f"""TODO: Sei [ruolo specifico del tuo agente].

Il tuo compito è [descrivi cosa deve fare].

Possibili esiti:
- ✅ ESITO POSITIVO — quando [condizione]
- ⚠️  ESCALATION — quando [condizione]
- ❌ ESITO NEGATIVO — quando [condizione]

Sequenza:
1. Chiama 'mio_tool_1' per [ottenere cosa]
2. Chiama 'mio_tool_2' per [verificare cosa]
3. Formula la decisione
4. Rispondi in italiano con la decisione motivata.

=== REGOLE DEL DOMINIO ===
{MIA_POLICY_TEXT}
=========================
"""

MIO_PROMPT = ChatPromptTemplate.from_messages([
    ("system", MIO_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Prompt template creato. Variabili:", MIO_PROMPT.input_variables)

# Checklist rapida
checks = [
    ("Ruolo specifico",         "TODO" not in MIO_SYSTEM_PROMPT.split("\n")[0]),
    ("Esiti elencati",           "ESITO" not in MIO_SYSTEM_PROMPT),
    ("Sequenza tool definita",   "Sequenza" in MIO_SYSTEM_PROMPT),
    ("Risposta in italiano",     "italiano" in MIO_SYSTEM_PROMPT.lower()),
    ("Regole incluse",           len(MIA_POLICY_TEXT.strip()) > 20),
]
print("\n📋 Checklist:")
for voce, ok in checks:
    print(f"   {'✅' if ok else '⚠️ '} {voce}")

---
## FASE 4 — Assembla il Tuo AgentExecutor

In [ ]:
# ============================================================
# FASE 4: ASSEMBLAGGIO
# ============================================================

MIO_TOOLS = [mio_tool_1, mio_tool_2]   # aggiungi altri tool se necessario

mio_agent = create_tool_calling_agent(llm, MIO_TOOLS, MIO_PROMPT)

mio_executor = AgentExecutor(
    agent   = mio_agent,
    tools   = MIO_TOOLS,
    verbose = True,
    max_iterations = 8,
    handle_parsing_errors = True,
)

print("✅ AgentExecutor pronto.")
print(f"   Tool: {[t.name for t in MIO_TOOLS]}")

---
## FASE 5 — Test: 3 Scenari con Esiti Diversi

In [ ]:
# ============================================================
# FASE 5: DEFINISCI I TUOI 3 SCENARI
# ============================================================

MIE_RICHIESTE_TEST = [
    {
        "label":        "Scenario 1 — Esito positivo",
        "richiesta":    "TODO: richiesta che porta all'esito positivo",
        "esito_atteso": "ESITO POSITIVO",
        "keywords":     ["approv", "ok", "positiv", "✅"]  # modifica con le parole chiave del tuo esito
    },
    {
        "label":        "Scenario 2 — Escalation",
        "richiesta":    "TODO: richiesta che porta all'escalation",
        "esito_atteso": "ESCALATION",
        "keywords":     ["escal", "manager", "approvaz", "⚠️"]
    },
    {
        "label":        "Scenario 3 — Esito negativo",
        "richiesta":    "TODO: richiesta che porta al rifiuto",
        "esito_atteso": "ESITO NEGATIVO",
        "keywords":     ["rifiut", "non possibi", "❌"]
    }
]

print("📋 Scenari definiti:")
for s in MIE_RICHIESTE_TEST:
    print(f"   • {s['label']} → {s['esito_atteso']}")

In [ ]:
# Esecuzione dei test
risposte = []

for scenario in MIE_RICHIESTE_TEST:
    print(f"\n{'═'*60}\n  {scenario['label']}\n{'═'*60}")
    result = mio_executor.invoke({"input": scenario["richiesta"]})
    risposte.append(result["output"])

In [ ]:
# Riepilogo risultati
print("\n" + "═" * 60)
print("  RISULTATI — CRITERI DI SUCCESSO GIORNO 3")
print("═" * 60)

tutti_ok = True
for scenario, risposta in zip(MIE_RICHIESTE_TEST, risposte):
    trovato = any(k.lower() in risposta.lower() for k in scenario["keywords"])
    esito   = "✅ OK" if trovato else "⚠️  Verifica manuale"
    if not trovato: tutti_ok = False
    print(f"\n  {scenario['label']}")
    print(f"  Verifica automatica: {esito}")

print("\n" + "═" * 60)
print("  🏆 OBIETTIVO RAGGIUNTO!" if tutti_ok else "  ⚠️  Leggi le risposte sopra e verifica manualmente.")
print("  Suggerimenti se il modello non risponde come atteso:")
print("  1. Migliora la docstring del tool (più specifica sul QUANDO)")
print("  2. Aggiungi la riga 'Rispondi SEMPRE in italiano.' al system prompt")
print("  3. Controlla che la sequenza operativa sia esplicita")
print("═" * 60)

---
## FASE 6 — Estensioni (Opzionali)

### Estensione A — Human Approval con `BaseCallbackHandler`

In [ ]:
# ESTENSIONE A — Human Approval
# Adatta il callback del Lab 3 al tuo caso d'uso
import datetime
from typing import Any

class MioHumanApprovalCallback(BaseCallbackHandler):
    """Callback Human Approval personalizzato."""

    def __init__(self, trigger_tool_name: str, auto_mode: bool = True):
        """trigger_tool_name: nome del tool che attiva il checkpoint."""
        self.trigger_tool = trigger_tool_name
        self.auto_mode    = auto_mode
        self.audit_log    = []

    def on_tool_start(self, serialized: dict, input_str: str, **kwargs: Any) -> None:
        self._current_tool = serialized.get("name", "")

    def on_tool_end(self, output: Any, **kwargs: Any) -> None:
        if self._current_tool != self.trigger_tool:
            return
        # TODO: personalizza il checkpoint per il tuo caso d'uso
        print("\n" + "═" * 50)
        print(f"  🔐 HUMAN APPROVAL — tool '{self.trigger_tool}' eseguito")
        print(f"  Output: {str(output)[:200]}")
        print("═" * 50)
        if self.auto_mode:
            print("  [AUTO_MODE] Approvato.")
            approved, note = True, "auto"
        else:
            dec  = input("  Approvi? [s/n]: ").strip().lower()
            approved = dec in ("s", "y", "si")
            note = input("  Nota: ").strip() or ("Approvato" if approved else "Rifiutato")
        self.audit_log.append({
            "tool": self.trigger_tool, "approved": approved,
            "note": note, "ts": datetime.datetime.now().isoformat()
        })

# TODO: Crea l'executor con il callback
# cb = MioHumanApprovalCallback(trigger_tool_name="mio_tool_2", auto_mode=True)
# mio_executor_con_approval = AgentExecutor(
#     agent=mio_agent, tools=MIO_TOOLS, verbose=True, callbacks=[cb]
# )
print("✅ MioHumanApprovalCallback definito. Decommenta per usarlo.")

### Estensione B — Terzo Tool

In [ ]:
# ESTENSIONE B — Terzo tool
@tool
def mio_tool_3(param: str) -> dict:
    """TODO: terzo tool per il tuo dominio."""
    return {"status": "OK", "risultato": f"Tool 3 output per {param}"}

# Aggiorna l'executor con il terzo tool:
# MIO_TOOLS_V2 = [mio_tool_1, mio_tool_2, mio_tool_3]
# mio_agent_v2 = create_tool_calling_agent(llm, MIO_TOOLS_V2, MIO_PROMPT)
# mio_executor_v2 = AgentExecutor(agent=mio_agent_v2, tools=MIO_TOOLS_V2, verbose=True)
print("✅ Terzo tool definito. Decommenta per aggiornare l'executor.")

### Estensione C — Agent-as-Tool
Crea un sotto-agente specializzato (es. un validatore delle regole) e wrappalo con `Tool.from_function()`.

In [ ]:
# ESTENSIONE C — Sotto-agente specializzato
MIO_SUB_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """TODO: sei un sotto-agente specializzato in [aspetto del tuo dominio].
Rispondi solo a domande su questo aspetto specifico. Rispondi in italiano."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# TODO: Crea il sotto-agente
# sub_agent_inner = create_tool_calling_agent(llm, [], MIO_SUB_AGENT_PROMPT)
# sub_executor    = AgentExecutor(agent=sub_agent_inner, tools=[], verbose=False)

# TODO: Wrappa come Tool
# mio_sub_agent_tool = Tool.from_function(
#     func        = lambda q: sub_executor.invoke({"input": q})["output"],
#     name        = "mio_sotto_agente",
#     description = "TODO: quando l'orchestratore deve delegare a questo sotto-agente?"
# )
print("✅ Schema sotto-agente pronto. Decommenta per implementare.")

---
## 📎 Scenario Alternativo — Assistente Richieste Ferie/Permessi

Se non hai un caso d'uso personale, usa questo scenario già implementato con LangChain.

In [ ]:
# ── Dati HR ──────────────────────────────────────────────────────────────
DIPENDENTI_HR = {
    "HR001": {"nome": "Luca Ferrari",  "team": "Engineering", "ferie_residue": 18, "permessi_ore": 24},
    "HR002": {"nome": "Sara Martini",  "team": "Marketing",   "ferie_residue": 5,  "permessi_ore": 4},
    "HR003": {"nome": "Pietro Romano", "team": "Sales",       "ferie_residue": 22, "permessi_ore": 16}
}

POLICY_FERIE = """
## Policy Ferie e Permessi
- Permessi orari brevi (<= 4h): approvazione automatica se ore disponibili.
- Ferie o permessi >= 8h: richiedono conferma del manager (escalation).
- Ferie residue < giorni richiesti: rifiuto automatico.
- Preavviso < 5 giorni lavorativi: escalation manager (salvo emergenze).
"""

# ── Tool HR ──────────────────────────────────────────────────────────────
@tool
def get_dipendente_hr(employee_id: str) -> dict:
    """Recupera i dati HR del dipendente: ferie residue, ore permesso, team.
    Chiamare SEMPRE prima di valutare qualsiasi richiesta di ferie o permesso. Badge es. 'HR001'."""
    if employee_id not in DIPENDENTI_HR:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    emp = DIPENDENTI_HR[employee_id]
    return {"status": "OK", "employee_id": employee_id, **emp}

@tool
def verifica_disponibilita_ferie(employee_id: str, giorni_richiesti: int) -> dict:
    """Verifica se il dipendente ha ferie residue sufficienti per i giorni richiesti.
    Chiamare per richieste di ferie. Parametri: employee_id (badge), giorni_richiesti (intero)."""
    if employee_id not in DIPENDENTI_HR:
        return {"status": "NOT_FOUND"}
    residue = DIPENDENTI_HR[employee_id]["ferie_residue"]
    return {
        "status": "OK",
        "ferie_residue": residue,
        "giorni_richiesti": giorni_richiesti,
        "disponibile": residue >= giorni_richiesti,
        "ferie_dopo": max(0, residue - giorni_richiesti)
    }

# ── Agente HR ─────────────────────────────────────────────────────────────
HR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", f"""Sei l'Assistente HR per la gestione di ferie e permessi.

Possibili esiti:
- ✅ APPROVAZIONE AUTOMATICA — permesso <= 4h con ore disponibili, o ferie con preavviso e giorni sufficienti.
- ⚠️  ESCALATION AL MANAGER — ferie con preavviso breve, permessi >= 8h.
- ❌ RIFIUTO — ferie residue insufficienti.

Sequenza:
1. Chiama 'get_dipendente_hr' per ferie residue e ore permesso
2. Se richiesta ferie: chiama 'verifica_disponibilita_ferie'
3. Confronta con la policy e decidi
4. Rispondi in italiano con la decisione motivata.

=== POLICY ===
{POLICY_FERIE}
=============
"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

hr_tools    = [get_dipendente_hr, verifica_disponibilita_ferie]
hr_agent    = create_tool_calling_agent(llm, hr_tools, HR_PROMPT)
hr_executor = AgentExecutor(agent=hr_agent, tools=hr_tools, verbose=True,
                            max_iterations=8, handle_parsing_errors=True)

print("✅ Agente HR pronto. Tool:", [t.name for t in hr_tools])

In [ ]:
# Test Scenario Alternativo — 3 casi HR

test_hr = [
    ("HR - Caso 1: Permesso breve",
     "Ciao, sono Luca Ferrari (HR001). Devo assentarmi 3 ore domani per una visita medica. Posso?"),
    ("HR - Caso 2: Ferie insufficienti",
     "Salve, sono Sara Martini (HR002). Vorrei prendere 10 giorni di ferie in agosto. È possibile?"),
    ("HR - Caso 3: Preavviso breve",
     "Ciao, sono Pietro Romano (HR003). Ho un'emergenza, ho bisogno di 3 giorni di ferie da domani."),
]

risposte_hr = []
for label, richiesta in test_hr:
    print(f"\n{'═'*60}\n  {label}\n{'═'*60}")
    r = hr_executor.invoke({"input": richiesta})
    risposte_hr.append(r["output"])
    print("📤 RISPOSTA:", r["output"])

---
### ✅ Recap Lab 5

**Hai completato il tuo MVP con LangChain:**
- Fase 1: dati del dominio
- Fase 2: `@tool` con docstring come schema implicito
- Fase 3: `ChatPromptTemplate` con ruolo, esiti e sequenza
- Fase 4: `create_tool_calling_agent` + `AgentExecutor`
- Fase 5: test su 3 scenari

**Porta con te per il Giorno 4:**
- I tuoi `@tool` decorati
- Il tuo `ChatPromptTemplate`
- Il tuo `AgentExecutor`

Nel Giorno 4 questi diventeranno **nodi di un grafo LangGraph** con stato esplicito, routing condizionale e cicli.